In [1]:
from typing import List, Optional, Union
from pydantic import BaseModel, Field, HttpUrl
from datetime import datetime

# --- Deployment Sub-Models ---
class CloudParams(BaseModel):
    type: str  # enum: azure_iotc, aws, custom
    app_url: Optional[str] = None

class Deployment(BaseModel):
    uuid: str
    display_name: str
    description: str
    last_update_time: Optional[str] = None
    last_deploy_result: Optional[str] = None
    cloud_params: CloudParams
    # Note: In your project file, leaf/gateway are direct children, 
    # though your schema expected a 'configuration' wrapper
    leaf: List[dict] = []
    gateway: List[dict] = []

# --- Model Sub-Models ---
class ModelMetadata(BaseModel):
    type: str  # enum: classifier, anomaly_detector, predictor
    classes: List[str]

class AIModel(BaseModel):
    uuid: str
    name: str
    description: Optional[str] = None
    creation_time: Optional[str] = None # Using str because your project file has nulls
    metadata: ModelMetadata
    dataset: dict
    target: dict
    training: Optional[dict] = None # Marked optional as project file uses 'data_sufficiency' instead

# --- Main Project Model ---
class AIProject(BaseModel):
    ai_project_name: str
    display_name: str
    description: str
    uuid: str
    version: str = Field(..., pattern=r"^[0-9].[0-9].[0-9]$")
    creation_time: Optional[str] = None
    last_update_time: Optional[str] = None
    models: List[AIModel]
    deployments: List[Deployment]

In [7]:
import json
from pydantic import ValidationError

# Load the raw JSON data
with open('ai_get_started_smart_asset_tracking_md_mlc.json', 'r') as f:
    raw_data = json.load(f)

try:
    # This single line validates AND converts the JSON into a Python Object
    project = AIProject(**raw_data)

    print(raw_data)
    
    print(f"Success! Loaded Project: {project.display_name}")
    print(f"Number of Models: {len(project.models)}")
    
except ValidationError as e:
    print("Validation failed!")
    print(e.json())


{'$schema': '/data/artifacts/.assets/schemas/ai_main_schema.json', 'schema_version': 'v11', 'ai_project_name': 'get_started_smart_asset_tracking_md_mlc', 'display_name': 'Smart Asset Tracking MD', 'description': "Classification of package conditions using the MLC to detect stationary orientation (upright or not) or moving and shaking states.\n\nWith this example, you can individually track the state of a physical item, such as a package, to monitor whether it is moving or being shaken too harshly, and whether it is still upright or not. Classification detects even brief events. You can clone and retrain the model to adapt classes to your use case, for example by tailoring the 'shaken' class to particularly strong motion or, conversely, by making it more sensitive for fragile items. To use this example, simply position the board on top of the item, ensuring it is secured firmly.", 'type': 'industrial', 'uuid': 'uuid_to_replace_project', 'version': '1.0.0', 'creation_time': None, 'last_u

TypeError: print() got an unexpected keyword argument '$schema'